In [1]:
import pandas as pd
import numpy as np
import glob
import yaml

import xml.etree.ElementTree as ET

In [2]:
with open("config_dataset_ffpe.yaml", "r") as stream:
    config_dataset = yaml.safe_load(stream)

metadata_path = config_dataset['metadata_path']
metadata_path

'data/metadata_ffpe.csv'

In [3]:
metadata = pd.read_csv(metadata_path)
metadata

,image_path,rna_path,case_id,sample_slide_id,sample_rna_id,sample_type,data_type_info,id_pair
0,TCGA-60-2712-01Z-00-DX1.97003dfc-4b37-4491-86e...,80ca4c12-21e8-45d1-8820-537b99bce32d.rna_seq.a...,TCGA-60-2712,TCGA-60-2712-01Z,TCGA-60-2712-01A,Primary,DX,0
1,TCGA-56-7221-01Z-00-DX1.f897f1ee-2796-4183-931...,545f9937-b128-4f44-8b12-ded0fb79bf3f.rna_seq.a...,TCGA-56-7221,TCGA-56-7221-01Z,TCGA-56-7221-01A,Primary,DX,1
2,TCGA-21-A5DI-01Z-00-DX1.E9123261-ADE7-468C-9E9...,9b86812f-b1ee-4b6d-9691-8587f2487c4a.rna_seq.a...,TCGA-21-A5DI,TCGA-21-A5DI-01Z,TCGA-21-A5DI-01A,Primary,DX,2
3,TCGA-43-7657-01Z-00-DX1.d8a5d257-c5ca-4192-b6a...,6df80c92-775e-4bcf-b2c7-6cbd7e147447.rna_seq.a...,TCGA-43-7657,TCGA-43-7657-01Z,TCGA-43-7657-01A,Primary,DX,3
4,TCGA-94-7033-01Z-00-DX1.43146ed9-30a5-420d-bd9...,23f1ad0c-c9d5-408f-bba8-1bb71364007b.rna_seq.a...,TCGA-94-7033,TCGA-94-7033-01Z,TCGA-94-7033-01A,Primary,DX,4
...,...,...,...,...,...,...,...,...
476,TCGA-66-2785-01Z-00-DX1.b9439ee1-d22b-4ccd-b53...,b5356a8b-9401-442f-a5dd-7e9273723a67.rna_seq.a...,TCGA-66-2785,TCGA-66-2785-01Z,TCGA-66-2785-01A,Primary,DX,476
477,TCGA-77-6843-01Z-00-DX1.5ced4995-81a1-4dfd-82b...,0b646082-9e64-4fb2-a9c9-e283d4c49852.rna_seq.a...,TCGA-77-6843,TCGA-77-6843-01Z,TCGA-77-6843-01A,Primary,DX,477
478,TCGA-66-2781-01Z-00-DX1.ed9ff5b7-7c66-4bf7-bfb...,59fd6a72-d96d-42af-8013-11f43af6eed5.rna_seq.a...,TCGA-66-2781,TCGA-66-2781-01Z,TCGA-66-2781-01A,Primary,DX,478
479,TCGA-39-5028-01Z-00-DX1.7994ec22-746d-4c30-813...,77b0bb2f-6e9f-4c67-b64c-98eebf0bd99d.rna_seq.a...,TCGA-39-5028,TCGA-39-5028-01Z,TCGA-39-5028-01A,Primary,DX,479


In [4]:
def load_last_vist_day(sample_id):
    try:
        path =  glob.glob(f'data/*/*/*.{sample_id}.xml')[0]
        tree = ET.parse(path)
        root = tree.getroot()
        
        # Define the variable you're searching for
        variable_name = "vital_status"
        
        # Search for the variable in the XML tree
        for elem in root.iter():
            if "vital_status" in elem.tag:
                status = elem.text
            if "days_to_death" in elem.tag:
                days_to_death = elem.text 
            if "days_to_last_followup" in elem.tag:
                days_to_last_followup = elem.text
    
        if status == "Alive":
            return (sample_id, status, days_to_last_followup)
        else:
            return (sample_id, status, days_to_death)
    except:
        return (sample_id, np.nan, np.nan)

In [5]:
survival_metadata = metadata.case_id.apply(lambda x: load_last_vist_day(x))
survival_metadata = pd.DataFrame([(r) for r in survival_metadata.values], columns=["case_id", "censored", "event_time"])
survival_metadata

,case_id,censored,event_time
0,TCGA-60-2712,Dead,274
1,TCGA-56-7221,NaN,NaN
2,TCGA-21-A5DI,Alive,979
3,TCGA-43-7657,NaN,NaN
4,TCGA-94-7033,NaN,NaN
...,...,...,...
476,TCGA-66-2785,Alive,60
477,TCGA-77-6843,Dead,2224
478,TCGA-66-2781,Alive,121
479,TCGA-39-5028,Dead,52


In [6]:
survival_metadata = survival_metadata.drop_duplicates('case_id')
survival_metadata = survival_metadata[~survival_metadata.event_time.isna()]
survival_metadata = survival_metadata[survival_metadata.event_time.astype(int) > 0]
survival_metadata

,case_id,censored,event_time
0,TCGA-60-2712,Dead,274
2,TCGA-21-A5DI,Alive,979
5,TCGA-60-2710,Alive,2024
6,TCGA-60-2695,Alive,642
10,TCGA-60-2698,Dead,311
...,...,...,...
476,TCGA-66-2785,Alive,60
477,TCGA-77-6843,Dead,2224
478,TCGA-66-2781,Alive,121
479,TCGA-39-5028,Dead,52


In [7]:
survival_metadata.censored.value_counts()

censored
Alive    222
Dead     167
Name: count, dtype: int64

In [8]:
survival_metadata.to_csv(f'{metadata_path.replace(".csv", "")}_survival.csv', index=False)